# NYC Yellow Taxi — 고요금(High-Fare) 예측 End2End 분석

**주제**: Pandas vs Polars 처리 성능 비교 기반 고요금(High-Fare) 운행 예측

`src/` 모듈(`io_compare`, `eda`, `visualize`, `stats_analysis`, `ml_pipeline`, `report`)을 그대로 불러와 대화형으로 실행합니다. 스크립트로 한 번에 돌리려면 저장소 루트에서 `python -m src.main`을 실행하세요.

In [1]:
import sys
from pathlib import Path

# 노트북은 notebooks/ 에서 실행되므로 저장소 루트를 sys.path에 추가한다.
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from IPython.display import display

from src.io_compare import load_pandas, run_benchmark
from src.eda import clean_trip_data, drop_duplicates, fare_descriptive_stats, missing_summary
from src.visualize import plot_benchmark_bar, plot_hourly_fare_line
from src.stats_analysis import correlation_matrix, congestion_ttest
from src.ml_pipeline import train_evaluate_save
from src.report import build_context, generate_report

DATA_PATH = REPO_ROOT / "data/raw/yellow_tripdata_2026-05.parquet"
OUTPUT_DIR = REPO_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. 데이터 준비 — 결측치·중복·기술통계 + Pandas vs Polars 벤치마크

In [2]:
raw = load_pandas(DATA_PATH)

na = missing_summary(raw)
print("결측치 비율(%):")
display(na[na > 0])

_, dup_removed = drop_duplicates(raw)
print(f"중복행 제거: {dup_removed}건")

fare_stats = fare_descriptive_stats(raw)
display(fare_stats)

결측치 비율(%):


passenger_count         23.35
RatecodeID              23.35
store_and_fwd_flag      23.35
congestion_surcharge    23.35
Airport_fee             23.35
dtype: float64

중복행 제거: 0건


,fare_amount,total_amount
count,4.090836e+06,4.090836e+06
mean,2.151320e+01,3.048514e+01
std,1.901303e+01,2.297497e+01
min,-9.500000e+02,-9.510000e+02
25%,1.000000e+01,1.764000e+01
50%,1.630000e+01,2.394000e+01
75%,2.680000e+01,3.495000e+01
max,5.525990e+03,5.530740e+03


In [3]:
bench = run_benchmark(str(DATA_PATH), number=1)
display(bench["timings"])

cleaned, clean_stats = clean_trip_data(raw)
clean_stats

operation   tool  seconds
       로딩 pandas 0.183162
       로딩 polars 0.147205
      필터링 pandas 0.189512
      필터링 polars 0.170096
   시간대별집계 pandas 0.070947
   시간대별집계 polars 0.023033


,operation,tool,seconds
0,로딩,pandas,0.183162
1,로딩,polars,0.147205
2,필터링,pandas,0.189512
3,필터링,polars,0.170096
4,시간대별집계,pandas,0.070947
5,시간대별집계,polars,0.023033


[정제] 원본=4090836 -> 규칙적용=3911730 -> IQR적용=3434526 (총 제거=656310건)


{'rows_before': 4090836,
 'rows_after_rules': 3911730,
 'rows_after_iqr': 3434526,
 'rows_removed': 656310,
 'trip_distance_bounds': (np.float64(-3.1750000000000003),
  np.float64(8.225000000000001)),
 'trip_duration_bounds': (np.float64(-12.966666666666669), np.float64(45.3))}

## 2. 시각화 — Seaborn 처리속도 바차트 + Plotly 시간대별 평균요금 라인차트

In [4]:
bar_fig = plot_benchmark_bar(bench["timings"], save_path=OUTPUT_DIR / "benchmark_speed.png")
display(bar_fig)

<Figure size 900x600 with 1 Axes>

In [5]:
line_fig = plot_hourly_fare_line(bench["hourly_fare"], save_path=OUTPUT_DIR / "hourly_fare_line.html")
line_fig.show()

## 3. 통계 분석 — 상관계수(가설1) + 혼잡시간대 t-test(가설2)

In [6]:
corr_result = correlation_matrix(cleaned)
display(corr_result["correlation"])

[상관분석] trip_distance-total_amount r=0.6722, trip_duration-total_amount r=0.7048 -> 가설1 채택(강한 양의 상관 확인) (기준: r>=0.5)
               trip_distance  trip_duration  total_amount
trip_distance       1.000000       0.683448      0.672190
trip_duration       0.683448       1.000000      0.704793
total_amount        0.672190       0.704793      1.000000


,trip_distance,trip_duration,total_amount
trip_distance,1.000000,0.683448,0.672190
trip_duration,0.683448,1.000000,0.704793
total_amount,0.672190,0.704793,1.000000


In [7]:
ttest_result = congestion_ttest(cleaned)
ttest_result["message"]

[t-test] 혼잡시간대(n=1089260, mean=24.71) vs 비혼잡시간대(n=2345266, mean=24.88): t=-12.2058, p=2.9033e-34 -> p<0.05, 통계적으로 유의미한 차이 있음 | 가설2 기각(혼잡시간대 평균요금이 더 높다고 보기 어려움)


'[t-test] 혼잡시간대(n=1089260, mean=24.71) vs 비혼잡시간대(n=2345266, mean=24.88): t=-12.2058, p=2.9033e-34 -> p<0.05, 통계적으로 유의미한 차이 있음 | 가설2 기각(혼잡시간대 평균요금이 더 높다고 보기 어려움)'

## 4. ML Pipeline — is_high_fare(고요금) 분류

In [8]:
pipeline_result = train_evaluate_save(cleaned, model_path=OUTPUT_DIR / "model.joblib")
{k: v for k, v in pipeline_result.items() if k != "confusion_matrix"}

[Pipeline] is_high_fare 임계값(total_amount>=29.82) | 학습 2747620건 / 평가 686906건
accuracy=0.8961 precision=0.7380 recall=0.9084 f1=0.8144
confusion_matrix=[[458867, 55596], [15804, 156639]]
저장=/Users/euntaehyeon/skala-python/skala-python-team/output/model.joblib, 재로딩 예측 일치=True


{'threshold': np.float64(29.82),
 'accuracy': 0.8960556466241378,
 'precision': 0.7380450915259029,
 'recall': 0.9083523251161253,
 'f1': 0.8143902172726281,
 'n_train': 2747620,
 'n_test': 686906,
 'model_path': '/Users/euntaehyeon/skala-python/skala-python-team/output/model.joblib',
 'reload_matches': True}

## 5. report.md 자동 생성

In [9]:
context = build_context(
    dataset_path=str(DATA_PATH.relative_to(REPO_ROOT)),
    rows_before=clean_stats["rows_before"],
    rows_after=clean_stats["rows_after_iqr"],
    missing_summary=na,
    duplicates_removed=dup_removed,
    fare_stats=fare_stats,
    benchmark_timings=bench["timings"],
    correlation_result=corr_result,
    ttest_result=ttest_result,
    pipeline_result=pipeline_result,
    chart_paths={
        "benchmark_bar": "output/benchmark_speed.png",
        "hourly_fare_line": "output/hourly_fare_line.html",
    },
)
generate_report(
    context,
    template_path=REPO_ROOT / "templates/report_template.md.j2",
    output_path=REPO_ROOT / "report.md",
)

[report] 생성 완료: /Users/euntaehyeon/skala-python/skala-python-team/report.md


PosixPath('/Users/euntaehyeon/skala-python/skala-python-team/report.md')

## 결론

- 가설1(운행시간·거리 ↔ 총요금 강한 양의 상관): 위 상관분석 결과 참고
- 가설2(혼잡시간대 평균요금 > 비혼잡시간대): 위 t-test 결과 참고
- 전체 결과는 저장소 루트의 `report.md`에서도 확인 가능합니다.